# ⚡ 100% 合規雲端 GPU 2K 寫實私房寫真工作站 (內建 Kaggle SSD 高速模型掛載 + 零外網依賴)

In [ ]:
import os
import json
import time
import random
import shutil

# ==================== 1. 絕對鎖定的核心人設 (踩在性感邊界、100% 合規不露點 + 真實毛孔冷白瓷肌) ====================
CORE_PERSONA = (
    "Intimate indoor high-fashion boudoir portrait photograph shot on 35mm Kodak Portra 400 analog film, "
    "three gorgeous 20-year-old French-Vietnamese mixed-race Eurasian women wearing alluring deep-V sheer tulle mesh and delicate floral lace lingerie bodysuits "
    "(delicate floral lace embroidery artfully covering the bust apex, tasteful non-nude editorial boudoir), "
    "extremely fair milk-white translucent porcelain skin with natural rosy flush, "
    "clearly visible microscopic skin pores and realistic unretouched skin texture (zero beauty filter, not plastic), "
    "slim 165cm 50kg hourglass bodies, large natural realistic soft F-cup breasts with natural gravity and deep cleavage."
)

# 雙重防線：
# 1) 嚴格封鎖露點/全裸/違規字眼 (保護帳號 100% 安全不越界)
# 2) 嚴格封鎖日常街頭T恤、牛仔外套、戶外場景與 AI 磨皮塑膠肌
DEFAULT_NEGATIVE_PROMPT = (
    "nude, nudity, naked, topless, exposed nipples, areola, explicit, nsfw, pornographic, genitalia, uncensored, "
    "t-shirt, shirt, blouse, jacket, denim jacket, coat, sweater, jeans, pants, street clothes, casual clothes, "
    "modest everyday clothing, outdoors, street, park, plaza, water, crowd, trees, afro, frizzy hair, "
    "tanned skin, dark skin, olive skin, airbrushed, smooth plastic skin, beauty filter, over-smoothed skin, "
    "waxy skin, shiny oiled skin, CGI, 3d render, doll, mannequin, wax figure, silicone implants, fake breasts, "
    "digital painting, anime, cartoon, deformed hands, extra fingers, mutated limbs, bad anatomy, fused bodies, "
    "identical clone faces, watermark, text, blurry, low quality"
)

# ==================== 2. 影片生成專用：合規性感「換裝 / 褪下外罩衫 / 整理蕾絲肩帶」動態 Prompt 池 ====================
VIDEO_CHANGING_CLOTHES_PROMPTS = [
    {
        "zh": "【合規換裝動感 A】緩緩將絲質晨袍從雙肩褪下，露出內搭深 V 透膚蕾絲連身衣、雙手輕調細肩帶與自然呼吸律動",
        "en": (
            "Three young Eurasian women in a sunlit bedroom changing clothes, slowly slipping loose silk morning robes "
            "off their shoulders to reveal alluring deep-V floral lace bodysuits underneath (remaining covered in elegant lace lingerie), "
            "gently adjusting their thin lace straps with their hands, natural soft chest breathing and upper body movement, "
            "warm alluring smiles at the camera, photorealistic live-action human motion, high skin detail."
        )
    },
    {
        "zh": "【合規換裝動感 B】在全身鏡前試穿極薄網紗蕾絲內衣、手指輕拉調整肩帶與腰身蕾絲、撥弄長捲髮並轉身展現沙漏曲線",
        "en": (
            "Three young Eurasian women wearing sheer tulle mesh and floral lace lingerie sets in front of a large mirror, "
            "gently pulling up and fitting delicate lace straps over their shoulders as they finish dressing, brushing long wavy hair back "
            "with their fingers, turning their torsos naturally to admire their hourglass curves in the mirror, realistic body movement and soft eye blinks."
        )
    },
    {
        "zh": "【合規換裝動感 C】解開外層透視雪紡罩衫換裝、展示內層精緻蕾絲馬甲與深邃事業線、在純白床單上親密微傾互動",
        "en": (
            "Three young Eurasian women sitting closely on crisp white bedsheets changing outfits, "
            "slowly untying and slipping off an outer sheer chiffon wrap to showcase their delicate floral lace bralettes and bodysuits underneath, "
            "leaning naturally toward each other with soft smiles, realistic hand gestures smoothing their lace lingerie, natural breathing and subtle body sway."
        )
    }
]

DEFAULT_VIDEO_NEGATIVE_PROMPT = (
    "nude, nudity, naked, topless, exposed nipples, areola, explicit, nsfw, stripping naked, "
    "static image, camera shake only, frozen body, stiff pose, t-shirt, street clothes, outdoors, "
    "distorted face, morphing features, extra limbs, deformed hands, blurry, low quality, watermark"
)

# ==================== 3. 帶中文標籤的隨機多樣化維度池 (精準踩在 ref_1 / ref_2 性感邊界、絕不露點越界) ====================
FACE_SHAPES = [
    {"zh": "精緻鵝蛋臉", "en": "delicate oval face with soft feminine features"},
    {"zh": "小巧心形臉", "en": "petite heart-shaped face with expressive eyes"},
    {"zh": "立體柔和 V 臉", "en": "soft V-line face with high cheekbones"},
    {"zh": "清純圓潤娃娃臉", "en": "youthful round baby-face with innocent charm"},
    {"zh": "法越混血立體輪廓", "en": "sculpted delicate jawline with classic French-Vietnamese features"},
    {"zh": "優雅菱形臉", "en": "graceful diamond-shaped face with deep-set eyes"}
]

HAIR_STYLES = [
    {"zh": "大波浪長捲髮", "en": "long flowing wavy hair"},
    {"zh": "及腰絲滑長直髮", "en": "silky straight waist-length hair"},
    {"zh": "法式下巴短鮑伯頭", "en": "chic chin-length French bob haircut"},
    {"zh": "隨性高丸子頭配碎髮", "en": "effortless messy high bun with loose face-framing tendrils"},
    {"zh": "鎖骨層次中長髮", "en": "shoulder-length layered collarbone cut"},
    {"zh": "浪漫半紮公主波浪捲", "en": "long romantic half-up waves"},
    {"zh": "空氣瀏海低馬尾", "en": "sleek low ponytail with wispy bangs"}
]

HAIR_COLORS = [
    {"zh": "純粹烏黑", "en": "jet black"},
    {"zh": "深濃縮咖啡棕", "en": "deep espresso brown"},
    {"zh": "暖栗褐", "en": "warm chestnut brown"},
    {"zh": "黑巧克力棕", "en": "dark chocolate brown"},
    {"zh": "焦糖冷灰棕", "en": "caramel ash brown"},
    {"zh": "蜂蜜紅棕", "en": "rich honey auburn"},
    {"zh": "柔霧黑棕", "en": "soft black-brown"}
]

MAKEUP_STYLES = [
    {"zh": "水光裸妝透亮唇", "en": "dewy bare-faced no-makeup look with glossy lips"},
    {"zh": "法式玫瑰腮紅配微雀斑", "en": "soft French rose tint on lips and cheeks with subtle freckles"},
    {"zh": "蜜桃水光玻璃肌", "en": "luminous glass-skin finish with peach-pink blush"},
    {"zh": "自然細眼線配珊瑚粉唇", "en": "delicate natural winged eyeliner and soft coral-pink lips"},
    {"zh": "晨間自然紅潤透亮肌", "en": "fresh natural rosy flush with clear hydrated skin"}
]

ATTIRE_STYLES = [
    {
        "zh": "【深 V 透膚網紗蕾絲連身衣】全透視薄紗腰身配精緻立體刺繡罩杯與高叉剪裁（對齊 ref_1 款式，性感不露點）",
        "category": "plunge_sheer_bodysuit",
        "needs_color": True,
        "en": "alluring deep-V plunging sheer tulle mesh and floral Chantilly lace bodysuit with high-cut leg openings, translucent mesh midriff and delicate floral lace embroidery covering the bust cups, showcasing full F-cup cleavage"
    },
    {
        "zh": "【法式薄紗蕾絲低胸內衣組】半透明網紗托胸內衣配細帶蕾絲低腰褲（對齊 ref_2 款式，凸顯事業線與平坦小腹）",
        "category": "sheer_lace_bralette_set",
        "needs_color": True,
        "en": "delicate sheer tulle mesh and floral lace demi-cup bralette with thin spaghetti straps and matching sheer lace low-rise bikini brief, artfully covering the bust apex while accentuating deep F-cup cleavage and toned waist"
    },
    {
        "zh": "【微透膚綁帶蕾絲馬甲連身衣】深 V 爆乳剪裁配全透明網紗側腰與細繩高叉設計",
        "category": "strappy_sheer_corset",
        "needs_color": True,
        "en": "sensual deep-plunge sheer mesh and scalloped floral lace corset bodysuit with thin strappy waist details and high-cut hips, floral lace cups covering the bust apex while revealing dramatic cleavage"
    },
    {
        "zh": "【半褪絲質薄紗罩衫配蕾絲內衣】肩頭微滑落的透明薄紗睡袍，內搭深 V 蕾絲三點式內衣組",
        "category": "silk_robe_and_lace_set",
        "needs_color": True,
        "en": "translucent gossamer chiffon negligee slipping slightly off her shoulders over a fitted deep-V floral lace bralette and sheer mesh panties, highlighting her full F-cup cleavage and hourglass curves"
    }
]

LINGERIE_COLORS = [
    {"zh": "珍珠白", "en": "pure pearl white"},
    {"zh": "午夜黑", "en": "midnight black"},
    {"zh": "勃根地深酒紅", "en": "deep burgundy wine red"},
    {"zh": "香檳裸膚色", "en": "champagne blush nude"},
    {"zh": "乾燥玫瑰粉", "en": "muted dusty rose pink"},
    {"zh": "柔霧薰衣草紫", "en": "soft lavender mist"},
    {"zh": "冰川淺藍", "en": "pale ice blue"},
    {"zh": "香檳象牙白", "en": "champagne ivory"}
]

POSES_AND_INTERACTIONS = [
    {
        "zh": "跪坐在柔軟純白床單上親密貼近（雙手自然撐在床面、微傾靠近彼此看鏡頭，對齊 ref_1 / ref_2）",
        "short_en": "kneeling closely together on crisp white bedsheets",
        "en": "kneeling and sitting closely together on soft white bedding, hands resting naturally on the bedsheets, leaning slightly toward one another looking softly at the camera"
    },
    {
        "zh": "全身站立於日式榻榻米親密互動（並肩貼近、輕觸彼此手臂與肩膀，對齊原始 Prompt）",
        "short_en": "standing close together full-body on Japanese tatami mats",
        "en": "standing close together facing the camera full body visible from head to toe, light touches on arms and shoulders, leaning slightly toward one another with shared soft smiles"
    },
    {
        "zh": "微側身親密依靠在床邊（一人輕靠對方肩頭、展現沙漏腰臀與深邃事業線，對齊 ref_2）",
        "short_en": "sitting closely together at three-quarter angles on a white bed",
        "en": "sitting closely together at slight three-quarter angles, leaning gently against each other's shoulders, showcasing hourglass curves, full F-cup cleavage, and warm eye contact"
    }
]

BACKGROUNDS_AND_LIGHTING = [
    {
        "zh": "日式傳統和室榻榻米 ＋ 背後大型全身鏡清晰反射美背與全身曲線 ＋ 障子紙窗自然柔光（原始 Prompt 場景）",
        "short_en": "in a Japanese washitsu tatami room with a large mirror behind them reflecting their backs, shoji window light",
        "en": "on Japanese tatami mats in a traditional washitsu room with a large clear mirror behind them clearly reflecting their full backs and bodies, natural window light from shoji screens"
    },
    {
        "zh": "巴黎極簡純白床單臥室 ＋ 暖白牆面與側邊柔和晨光（完全對齊 ref_1 & ref_2 參考圖場景）",
        "short_en": "on crisp white linen bedsheets in a minimalist bedroom against a clean warm-white wall, soft morning window light",
        "en": "on crisp white linen bedsheets in a minimalist sunlit bedroom against a warm off-white wall, soft diffused morning window light casting gentle shadows"
    },
    {
        "zh": "北歐落地窗陽光臥室 ＋ 無框大型全身鏡反射背影 ＋ 黃金時刻柔光",
        "short_en": "in a bright bedroom studio with a large full-length mirror reflecting their backs, golden-hour window light",
        "en": "in a bright Scandinavian bedroom with floor-to-ceiling windows and a large full-length frameless mirror reflecting their backs, bathed in soft golden-hour sunlight"
    }
]


def generate_diversified_prompt(seed=None, **kwargs):
    """
    產生「精準踩在性感邊界、100% 合規不露點、且不超過 SDXL 77-Token 視窗」的雙層寫實 Prompt。
    """
    rng = random.Random(seed)
    image_seed = seed if seed is not None else rng.randint(10000000, 99999999)

    faces = rng.sample(FACE_SHAPES, 3)
    hairs = rng.sample(HAIR_STYLES, 3)
    colors = rng.sample(HAIR_COLORS, 3)
    makeups = rng.sample(MAKEUP_STYLES, 3)
    lingerie_colors = rng.sample(LINGERIE_COLORS, 3)
    attires = rng.sample(ATTIRE_STYLES, 3)

    pose = rng.choice(POSES_AND_INTERACTIONS)
    bg = rng.choice(BACKGROUNDS_AND_LIGHTING)
    vid_motion = rng.choice(VIDEO_CHANGING_CLOTHES_PROMPTS)

    women_descriptions = []
    women_metadata = []
    positions_en = ["left woman", "center woman", "right woman"]
    positions_zh = ["左邊女孩", "中間女孩", "右邊女孩"]

    short_color_list = ", ".join([c["en"] for c in lingerie_colors])

    for idx in range(3):
        att = attires[idx]
        attire_en = f"wearing a {lingerie_colors[idx]['en']} {att['en']}"
        attire_zh = f"{lingerie_colors[idx]['zh']} — {att['zh']}"

        desc = (
            f"The {positions_en[idx]} has a {faces[idx]['en']}, {colors[idx]['en']} {hairs[idx]['en']}, "
            f"{makeups[idx]['en']}, {attire_en}."
        )
        women_descriptions.append(desc)
        women_metadata.append({
            "position_zh": positions_zh[idx],
            "face_zh": faces[idx]["zh"],
            "hair_color_zh": colors[idx]["zh"],
            "hair_style_zh": hairs[idx]["zh"],
            "makeup_zh": makeups[idx]["zh"],
            "attire_zh": attire_zh,
            "category": att["category"]
        })

    # 控制在 ~64 tokens 內，同時包含：深 V 透膚網紗蕾絲連身衣 + 蕾絲遮蔽重點部位 (non-nude) + 冷白毛孔瓷肌 + F罩杯深邃事業線 + 室內床單/和室鏡子
    clip_short_prompt = (
        f"Intimate indoor boudoir photo of three gorgeous 20yo French-Vietnamese mixed-race women "
        f"wearing deep-V sheer tulle mesh and floral lace bodysuits in {short_color_list} (floral lace covering bust apex, non-nude), "
        f"extremely fair milk-white porcelain skin, rosy flush, visible skin pores, "
        f"165cm 50kg slim hourglass bodies, large natural soft F-cup cleavage, "
        f"{pose['short_en']} {bg['short_en']}."
    )

    full_prompt = (
        f"{clip_short_prompt} "
        f"{' '.join(women_descriptions)} "
        f"All three are {pose['en']}, {bg['en']}. "
        "Shot on 35mm Kodak Portra 400 film, unretouched realistic skin pores and soft natural breast cleavage texture, soft window light, tasteful high-fashion boudoir."
    )

    summary_lines = [
        f"🔑 隨機種子 (Seed): {image_seed}",
        f"🏞️ 場景與光線: {bg['zh']}",
        f"💃 姿勢與互動: {pose['zh']}",
        f"🎬 對應合規換裝動態 Prompt: {vid_motion['zh']}"
    ]
    for w in women_metadata:
        summary_lines.append(
            f"👩 {w['position_zh']}: {w['face_zh']} | {w['hair_color_zh']}{w['hair_style_zh']} | {w['makeup_zh']} | {w['attire_zh']}"
        )

    metadata = {
        "seed": image_seed,
        "core_persona_zh": "20歲法越混血女性 × 3、無磨皮真實毛孔質感、冷白透亮瓷肌、165cm 50kg 自然柔軟 F 罩杯沙漏體態（深 V 透膚蕾絲精準遮蔽重點、性感不越界）",
        "background_zh": bg["zh"],
        "pose_zh": pose["zh"],
        "video_motion_zh": vid_motion["zh"],
        "video_motion_prompt": vid_motion["en"],
        "video_negative_prompt": DEFAULT_VIDEO_NEGATIVE_PROMPT,
        "women": women_metadata,
        "summary_text": "\n".join(summary_lines),
        "clip_short_prompt": clip_short_prompt,
        "full_prompt": full_prompt,
        "negative_prompt": DEFAULT_NEGATIVE_PROMPT
    }
    return full_prompt, metadata



import sys, gc, glob
import torch, psutil
from PIL import Image, ImageFilter

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
gpu_name = f"{torch.cuda.get_device_name(0)} (x{gpu_count})" if gpu_count > 0 else "CPU"
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2) if gpu_count > 0 else 0
ram_gb = round(psutil.virtual_memory().total / (1024**3), 2)
print(f"✅ 硬體檢測完成 | GPU: {gpu_name} | 單卡 VRAM: {vram_gb} GB | System RAM: {ram_gb} GB")

# 自動搜尋 Kaggle 內部高速掛載的 RealVisXL Diffusers 模型目錄 (免外網、10GB/s 秒速載入)
local_model_dir = None
for cand in glob.glob("/kaggle/input/**/model_index.json", recursive=True):
    d = os.path.dirname(cand)
    if os.path.exists(os.path.join(d, "unet")) and os.path.exists(os.path.join(d, "text_encoder")):
        local_model_dir = d
        break

from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

if local_model_dir:
    print(f"🚀 [極速本地 SSD 掛載] 發現 Kaggle 內部寫實模型: {local_model_dir}")
    pipe = StableDiffusionXLPipeline.from_pretrained(
        local_model_dir, torch_dtype=torch.float16, variant="fp16", local_files_only=True
    ).to("cuda")
    engine_label = "RealVisXL (SDXL Dual-CLIP 30-Steps + Kaggle Local SSD)"
else:
    print("🌐 正在從 Hugging Face 載入 SG161222/RealVisXL_V5.0...")
    pipe = StableDiffusionXLPipeline.from_pretrained(
        "SG161222/RealVisXL_V5.0", torch_dtype=torch.float16, variant="fp16"
    ).to("cuda")
    engine_label = "RealVisXL V5.0 (SDXL Dual-CLIP 30-Steps)"

pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, use_karras_sigmas=True, algorithm_type="sde-dpmsolver++"
)
pipe.vae.enable_tiling()

image_times = []
image_peak_vram = 0.0
max_res_str = "1536x2304"
human_ts = time.strftime("%Y-%m-%d %H:%M:%S")
num_imgs = 4

for i in range(num_imgs):
    nat_w, nat_h = (1024, 1536) if i == num_imgs - 1 else (960, 1440)
    out_w, out_h = (1536, 2304) if i == num_imgs - 1 else (1440, 2160)
    full_prompt, recipe = generate_diversified_prompt()

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    gen = torch.Generator("cuda").manual_seed(recipe["seed"])
    img = pipe(
        prompt=recipe["clip_short_prompt"],
        prompt_2=full_prompt,
        negative_prompt=recipe.get("negative_prompt", DEFAULT_NEGATIVE_PROMPT),
        negative_prompt_2=recipe.get("negative_prompt", DEFAULT_NEGATIVE_PROMPT),
        width=nat_w, height=nat_h,
        num_inference_steps=30,
        guidance_scale=5.5,
        generator=gen
    ).images[0]

    if img.size != (out_w, out_h):
        img = img.resize((out_w, out_h), Image.LANCZOS)
    img = img.filter(ImageFilter.UnsharpMask(radius=1.0, percent=45, threshold=3))

    dt = round(time.time() - t0, 2)
    pk = round(torch.cuda.max_memory_allocated() / (1024**3), 2)
    image_times.append(dt)
    image_peak_vram = max(image_peak_vram, pk)
    max_res_str = f"{out_w}x{out_h}"

    recipe["timestamp"] = human_ts
    recipe["model_engine"] = engine_label
    recipe["elapsed_sec"] = dt
    recipe["peak_vram_gb"] = pk
    recipe["resolution"] = f"{out_w}x{out_h}"

    fname = f"portrait_{i+1}.jpg"
    rname = f"portrait_{i+1}_recipe.json"
    img.save(fname, quality=96)
    with open(rname, "w", encoding="utf-8") as rf:
        json.dump(recipe, rf, ensure_ascii=False, indent=2)

    print(f"\n================ 🎲 第 {i+1} 張 [{out_w}x{out_h}] 耗時: {dt}s | 峰值 VRAM: {pk} GB ================")
    print(recipe["summary_text"])

bench_report = {
    "gpu_name": gpu_name,
    "vram_gb": vram_gb,
    "ram_gb": ram_gb,
    "avg_image_time_sec": round(sum(image_times)/len(image_times), 2) if image_times else 0,
    "image_peak_vram_gb": image_peak_vram,
    "max_resolution_tested": max_res_str,
    "avg_video_time_sec": 0,
    "video_peak_vram_gb": 0,
    "video_model": "已關閉 (純生圖極速模式)"
}
with open("benchmark_report.json", "w", encoding="utf-8") as bf:
    json.dump(bench_report, bf, ensure_ascii=False, indent=2)

print("\n🏆 雲端 2K 寫實生圖全部完成！")
